# Guardian of Truth v5 — сигналы для выбора текстового ранжировщика

**Что считает.** Одна модель Qwen3.6-35B-A3B-FP8 (как на платформе), без рассуждений, один токен на вопрос:
- `p_base` — промпт бейзлайна организаторов;
- `p_v1`, `p_v2` — старые судьи (запросы один в один как в v2.1 — сверим со старыми дампами);
- текстовые атомы — 6 узких вопросов к ходам без вызовов (новое);
- атомы по вызовам (стратегия C) — для ходов с вызовами.

**Наборы:** dev (valid + пул, 430 ходов) и новые диалоги τ² из ноутбука `guardian_v5_tau2gen`
(`MyDrive/guardian_of_truth/v5_tau2gen/examples_v5.parquet`). **Запускай после окончания генерации**
(отдельная сессия Colab). Если файла ещё нет, шаг 7 скажет об этом — перезапусти его потом.

**Среда:** Colab, **A100 80GB**, High-RAM. Секрет `HF_TOKEN` желателен.

**Оценка:** ≈ 60–75 мин ≈ **8–9 CU** (модель качается ~5–10 мин, dev ~10 мин, новые диалоги ~30–45 мин).
RAM ≈ 15 ГБ, диск ≈ 40 ГБ.

**Чекпоинты:** каждая оценка пишется в jsonl на Диске (`MyDrive/guardian_of_truth/v5_signals/`); после
обрыва перезапусти всё — досчитается только недостающее.

**Результат:** `MyDrive/guardian_of_truth/v5_signals/results_v5_signals.zip` — пришли его Claude.

In [1]:
# 1. Диск, папки, GPU  (⏱ <1 мин)
import os, sys, json, time, subprocess, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/guardian_of_truth/v5_signals'); OUT.mkdir(parents=True, exist_ok=True)
GENERATED = Path('/content/drive/MyDrive/guardian_of_truth/v5_tau2gen/examples_v5.parquet')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'],
                     capture_output=True, text=True).stdout)

Mounted at /content/drive
name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB



In [2]:
# 2. vLLM 0.19.1 — как на платформе  (⏱ 3–6 мин в первый раз)
import importlib.metadata as md
try:
    vllm_version = md.version('vllm')
except md.PackageNotFoundError:
    vllm_version = None
if vllm_version != '0.19.1':
    !pip -q install vllm==0.19.1
print('vllm', md.version('vllm'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/1

In [3]:
# 3. Модель Qwen3.6-35B-A3B-FP8  (⏱ 5–10 мин в первый раз)
from huggingface_hub import snapshot_download
MODEL = Path('/content/models/user')
if not (MODEL / '.complete').exists():
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as error:
        print('HF_TOKEN не найден, качаю без него:', error)
    t = time.time()
    snapshot_download('Qwen/Qwen3.6-35B-A3B-FP8', local_dir=str(MODEL),
                      allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja', 'merges.txt'])
    (MODEL / '.complete').write_text('Qwen/Qwen3.6-35B-A3B-FP8')
    print(f'скачано за {time.time() - t:.0f} с')
print(sorted(p.name for p in MODEL.iterdir())[:10])

HF_TOKEN не найден, качаю без него: Secret HF_TOKEN does not exist.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 53 files:   0%|          | 0/53 [00:00<?, ?it/s]

скачано за 98 с
['.cache', '.complete', 'chat_template.jinja', 'config.json', 'configuration.json', 'generation_config.json', 'layers-0.safetensors', 'layers-1.safetensors', 'layers-10.safetensors', 'layers-11.safetensors']


In [4]:
# 4. Код (veritas + старые судьи) и dev-набор из valid  (⏱ <1 мин)
import base64, io, tarfile
CODE_B64 = '...'
FRESH = Path('/content/fresh'); FRESH.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode='r:gz') as archive:
    archive.extractall(FRESH)
!cd {FRESH} && {sys.executable} dev/make_pool.py --valid valid.parquet --out dev/data
print(sorted(str(p.relative_to(FRESH)) for p in FRESH.glob('*/*'))[:12])

/tmp/ipykernel_762/1807955828.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(FRESH)


pool 384  dev 430  dialogs 40
['dev/data', 'dev/legacy_v2', 'dev/make_pool.py', 'dev/score_signals.py', 'src/veritas']


In [5]:
# 5. Сервер vLLM для судей  (⏱ 4–6 мин)
import urllib.request
subprocess.run('pkill -f "vllm.entrypoints.openai.api_server" || true', shell=True)   # на случай перезапуска ячейки
time.sleep(5)
log = open('/content/vllm_judge.log', 'w')
JUDGE = subprocess.Popen([sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', str(MODEL),
                          '--served-model-name', 'm', '--port', '8000', '--max-model-len', '131072',
                          '--gpu-memory-utilization', '0.90', '--max-num-seqs', '128', '--enable-prefix-caching',
                          '--language-model-only'], stdout=log, stderr=subprocess.STDOUT)
started = time.time()
while True:
    if JUDGE.poll() is not None:
        print(open('/content/vllm_judge.log').read()[-4000:]); raise RuntimeError('vLLM упал')
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=5); break
    except Exception:
        time.sleep(5)
print(f'vLLM готов за {time.time() - started:.0f} с')

vLLM готов за 325 с


In [6]:
# 6. Сигналы на dev (430 ходов)  (⏱ 8–12 мин)
SIGNALS = OUT / 'signals'
!cd {FRESH} && {sys.executable} dev/score_signals.py --input dev/data/dev.parquet --name dev \
    --base-url http://127.0.0.1:8000/v1 --out {SIGNALS}

dev: ходов 430, без вызовов 188, с вызовами 205, сработали проверки 37
[veritas 13:24:57] base |                              | 1/430 за 7 с, осталось ≈2951 с
[veritas 13:25:13] base |████████                      | 125/430 за 22 с, осталось ≈54 с
[veritas 13:25:28] base |███████████                   | 165/430 за 37 с, осталось ≈60 с
[veritas 13:25:43] base |███████████████               | 215/430 за 52 с, осталось ≈52 с
[veritas 13:25:58] base |████████████████████████      | 354/430 за 67 с, осталось ≈14 с
[veritas 13:26:03] base |██████████████████████████████| 430/430 за 73 с, осталось ≈0 с
  base: 430 оценок, 72.8 с, ошибок HTTP 0
[veritas 13:26:04] v1 |                              | 1/430 за 1 с, осталось ≈373 с
[veritas 13:26:20] v1 |███████                       | 101/430 за 16 с, осталось ≈53 с
[veritas 13:26:35] v1 |██████████                    | 153/430 за 31 с, осталось ≈57 с
[veritas 13:26:50] v1 |█████████████                 | 196/430 за 47 с, осталось ≈56 с
[veritas 

In [7]:
# 7. Сигналы на новых диалогах τ²  (⏱ 30–45 мин)
if GENERATED.exists():
    !cd {FRESH} && {sys.executable} dev/score_signals.py --input {GENERATED} --name tau2 \
        --base-url http://127.0.0.1:8000/v1 --out {SIGNALS}
else:
    print('examples_v5.parquet ещё нет — дождись конца генерации и перезапусти эту ячейку')

tau2: ходов 2810, без вызовов 1677, с вызовами 1117, сработали проверки 16
[veritas 13:32:49] base |                              | 1/2810 за 3 с, осталось ≈8407 с
[veritas 13:33:04] base |█                             | 149/2810 за 18 с, осталось ≈324 с
[veritas 13:33:19] base |███                           | 288/2810 за 33 с, осталось ≈291 с
[veritas 13:33:35] base |████                          | 444/2810 за 49 с, осталось ≈260 с
[veritas 13:33:50] base |██████                        | 594/2810 за 64 с, осталось ≈239 с
[veritas 13:34:09] base |██████                        | 604/2810 за 83 с, осталось ≈302 с
[veritas 13:34:25] base |██████                        | 612/2810 за 98 с, осталось ≈353 с
[veritas 13:34:40] base |██████                        | 651/2810 за 113 с, осталось ≈376 с
[veritas 13:34:55] base |███████                       | 695/2810 за 129 с, осталось ≈393 с
[veritas 13:35:10] base |███████                       | 733/2810 за 144 с, осталось ≈409 с
[veritas 13:35

In [8]:
# 8. Архив результатов → пришли его Claude  (⏱ <1 мин)
archive = OUT / 'results_v5_signals.zip'
!cd {OUT} && rm -f {archive} && zip -q -r {archive} signals
print(archive, f'{archive.stat().st_size / 1e6:.1f} МБ')
JUDGE.terminate()

/content/drive/MyDrive/guardian_of_truth/v5_signals/results_v5_signals.zip 0.5 МБ
